# IEEE-CIS TCN Benchmark on Colab

Notebook nay dung de benchmark nhanh nhanh `TCN` standalone tren Google Colab truoc khi xem xet fusion vao pipeline IEEE chinh.

## Runtime

Trong Colab, dat `Runtime -> Change runtime type -> Hardware accelerator -> GPU` truoc khi chay.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Repo config
GITHUB_REPO = 'https://github.com/Thanh-000/FDB1-DoAn.git'
REPO_PARENT = '/content/drive/MyDrive'
REPO_NAME = 'FDB1-DoAn'
REPO_DIR = f'{REPO_PARENT}/{REPO_NAME}'

# Dataset zip config, aligned with the main IEEE notebook
ZIP_PATH = '/content/drive/MyDrive/MVS_XAI_Data/ieee-fraud-detection.zip'
EXTRACT_DIR = '/content/ieee-fraud-detection'

# Benchmark config
FOLD_INDEX = 0
N_SPLITS = 5
EPOCHS = 20
HIDDEN_DIM = 64
WINDOW = 10


In [ ]:
import os
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

print('REPO_DIR =', REPO_DIR)


In [ ]:
import os
import zipfile
import glob

if ZIP_PATH and not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)

txn_files = glob.glob(f'{EXTRACT_DIR}/**/train_transaction.csv', recursive=True)
id_files = glob.glob(f'{EXTRACT_DIR}/**/train_identity.csv', recursive=True)

print('txn:', txn_files[:1])
print('id :', id_files[:1])

if not txn_files or not id_files:
    raise FileNotFoundError('Could not locate train_transaction.csv/train_identity.csv after extraction')

DATA_DIR = os.path.dirname(txn_files[0])
print('DATA_DIR =', DATA_DIR)


In [ ]:
%cd "$REPO_DIR"
!pwd
!ls
!ls sequence


In [ ]:
import sys
import torch
import sklearn
import pandas as pd

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('sklearn:', sklearn.__version__)
print('pandas:', pd.__version__)


In [ ]:
cmd = [
    'python', 'run_ieee_tcn.py',
    '--data-dir', DATA_DIR,
    '--fold-index', str(FOLD_INDEX),
    '--n-splits', str(N_SPLITS),
    '--window', str(WINDOW),
    '--epochs', str(EPOCHS),
    '--hidden-dim', str(HIDDEN_DIM),
]

print('Running:')
print(' '.join(cmd))


In [ ]:
!python run_ieee_tcn.py \
  --data-dir "$DATA_DIR" \
  --fold-index "$FOLD_INDEX" \
  --n-splits "$N_SPLITS" \
  --window "$WINDOW" \
  --epochs "$EPOCHS" \
  --hidden-dim "$HIDDEN_DIM"
